First Create the silver Schema


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbw_gitarchive.silver

In [0]:
%sql
SHOW SCHEMAS

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
#Setting up the params
source_table = "dbw_gitarchive.bronze.raw_github_events"
target_table = "dbw_gitarchive.silver.github_events"
checkpoint_path = "abfss://silver@stgithubarchive.dfs.core.windows.net/checkpoints/github_events"

Since thee params are set, starting the streaming


In [0]:
bronze_stream = spark.readStream.format("delta").table(source_table)

In [0]:
%sql
SELECT * FROM dbw_gitarchive.bronze.raw_github_events LIMIT 10

In [0]:
silver_stream_df = bronze_stream.select(\
    col("id").cast(StringType()).alias("event_id"),\
    col("type").alias("event_type"),\
    col("actor"),\
    col("repo"),\
    col("payload"),\
    col("public").cast("boolean").alias("is_public"),\
    col("org"),\
    col("source_file"),\
    col("created_at").cast("timestamp").alias("event_created_at"),\
    col("event_date"),\
    current_timestamp().alias("processed_at"))
    
    
    

        
    

Since the we have selected the data that we are taking to silver layer, let's write them using auto loader

In [0]:
#Using the declared parameters
query = silver_stream_df.writeStream.format("delta").\
    outputMode("append").\
    option("checkpointLocation", checkpoint_path).\
    trigger(availableNow = True).\
    toTable(target_table)

In [0]:
%sql
SELECT COUNT(*) FROM dbw_gitarchive.silver.github_events